In [ ]:
from typing import List

In [ ]:
import tensorflow as tf

# List available hardware
devices: List = tf.config.list_physical_devices()
print(f"Devices found: {devices}")

In [ ]:
gpu_devices: List = tf.config.list_physical_devices("GPU")

if gpu_devices:
    print(f"GPU devices found: {gpu_devices}")

    for each_gpu_device in gpu_devices:
        tf.config.experimental.set_memory_growth(each_gpu_device, True)

else:
    print("Running on Local CPU")

## Remarks: How to Fetch the Dataset

We will use `Kesejahteraan Pekerja Indonesia` dataset from [Kaggle](https://www.kaggle.com/datasets/rezkyyayang/pekerja-sejahtera) for this project

### Dataset Description

Messr Rezky Yayang Hamid curated this data from BPS in 2023. The dataset contains several metrics that gauge workers' well-being around Indonesia. The metrics included in this dataset consists of:
* upah.df.csv - this dataset gives information on the average hourly wage of workers grouped by provinces and years
* ump.df.csv - this dataset gives information about `upah minimum provinsi (UMP)`, which is the minimum wage grouped by provinces and years
* gk.df.csv = this dataset gives information about 'Garis Kemiskinan per kapita', which is poverty line on per capita basis grouped by province, year, survey period, type of outlay, and region of domicile
* peng.df.csv - this dataset gives information about the average spending on per capita basis grouped by province, year, type of outlay, and region of domicile


### Authentication

Authentication is needed to download this dataset from Kaggle

First, you will need a Kaggle account. You can sign up [here](https://www.kaggle.com/account/login)

After login, you can download your Kaggle API token at [here](https://www.kaggle.com/settings) by clicking on the "Generate new Token" button under the "API" section.

You have several options to authenticate:

Option 1: kagglehub.login()

```Python
import kagglehub

kagglehub.login()
```
This will prompt you to enter your Kaggle API token.

Option 2: Environment Variable

```Bash
export KAGGLE_API_TOKEN=xxxxxxxxxxxxxx # Coped from the settings UI
```

Option 3: API Token File

Store your Kaggle API token obtained from your [setting page](https://www.kaggle.com/settings) to a file located in `~/.kaggle/access_token`

Option 4: Google Colab Secret

If you are running this notebook from Google Cloud, then you can store your API token in a Colab secret named `KAGGLE_API_TOKEN`.

Instructions on adding secrets in Colab can be found [here](https://www.googlecloudcommunity.com/gc/Cloud-Hub/How-do-I-add-secrets-in-Google-Colab-Enterprise/m-p/784866)

Option 5: Legacy API credentials file

From your [Kaggle Account Setting Page](https://www.kaggle.com/settings), under the "Legacy API Credentials", click on the "Create Legeacy API Key" button to generate a `kaggle.json` file and store it at `~/.kaggle/kaggle.json`

In [ ]:
# Authenticate Kaggle

import os
from pathlib import Path


import kagglehub

is_authenticated: bool = False

api_key_path: Path = Path("~/.kaggle/kaggle.json").expanduser()
is_api_key_exists: bool = True if api_key_path.exists() else False

api_token_path: Path = Path("~/.kaggle/access_token").expanduser()
is_api_token_exists: bool = True if api_token_path.exists() else False

## Check if running in Colab
def is_colab() -> bool:
    """
    Small utility function to check if this notebook is running on Colab
    :return: Boolean indication if this script is running on Colab
    :rtype bool
    """
    try:
        import google.colab
        return True
    except ImportError:
        return False

is_colab_secret_exists: bool = False
if is_colab():
    from google.colab import userdata

    try:
        is_colab_secret_exists = True if userdata.get("KAGGLE_API_TOKEN") else False
    except userdata.SecretNotFoundError:
        is_colab_secret_exists = False

is_token_env_exists: bool = True if os.getenv("KAGGLE_API_TOKEN") else False

is_token_via_login: bool = False
if all([
    not is_api_key_exists
    , not is_api_token_exists
    , not is_colab_secret_exists
    , not is_token_env_exists
]):
    # if everything above fails, then login interactively
    # this will ask you to input the Kaggle API Token from a UI
    kagglehub.login()
    is_token_via_login = True


if any([
    is_api_key_exists
    , is_api_token_exists
    , is_token_env_exists
    , is_colab_secret_exists
    , is_token_via_login
]):
    is_authenticated = True
    print("Kagglehub has been authenticated")

else:
    raise Exception(" 🛑 Kagglehub fails to authenticate")

## Remarks: Download the Dataset

In this section, we will download the dataset from Kaggle.

We want to detect if any `.csv` files already present under `dataset/` folder.

If not, then we: (1) would download the dataset from Kaggle, (2) copy the downloaded data from Kaggle's `cache/` folder into the project's `dataset/` folder. If `.csv` files already present in the `dataset/` folder, then we will simply load the data from the `dataset/` folder.

This way, we can avoid downloading the dataset every time we run this notebook.

In [ ]:
# Download Data Smartly
# If the data is not yet downloaded, we will first download the data from Kaggle to `dataset` folder, and then load them to pandas
# Else, we will load the data from the `dataset` folder directly to pandas

from pathlib import Path
import shutil

import kagglehub

kaggle_dataset_handler_str: str = "rezkyyayang/pekerja-sejahtera"

project_root: Path = Path.cwd().expanduser()
dataset_folder: Path = project_root / "dataset"

dataset_folder.mkdir(parents=True, exist_ok=True)

available_files = sorted(each_found_file.name for each_found_file in dataset_folder.glob("*.csv"))

if available_files:
    print("Dataset already exists locally. Using files from `dataset` folder.")
    print("Available files:")
    print(available_files)
else:
    downloaded_path_str: str = kagglehub.dataset_download(
        handle=kaggle_dataset_handler_str
    )
    downloaded_path: Path = Path(downloaded_path_str)
    downloaded_files = sorted(each_csv_file_path.name for each_csv_file_path in downloaded_path.glob("*.csv"))

    print("Downloaded files:")
    print(downloaded_files)

    for each_downloaded_file in downloaded_files:
        shutil.copy2(
            downloaded_path / each_downloaded_file,
            dataset_folder / each_downloaded_file,
        )

    available_files = sorted(each_found_file.name for each_found_file in dataset_folder.glob("*.csv"))

## Remarks: Load the Dataset into Pandas DataFrame

In this section, we would simply load the dataset into Pandas DataFrame by calling `pd.read_csv()` method

We will also glimpse the number of rows, and if there are any null values we need to handle later.

For this inspection, we will use the `info()` method of Pandas DataFrame.


In [ ]:
import pandas as pd

# Load upah_df
upah_path: Path =

upah_df: pd.DataFrame =

print("upah_df.info():")
upah_df.info()
print("##"*10)

# Load ump_df
ump_path: Path =
ump_df: pd.DataFrame =

print("ump_df.info():")
ump_df.info()
print("##"*10)

# Load peng_df
peng_path: Path =
peng_df: pd.DataFrame =

print("peng_df.info():")
peng_df.info()
print("##"*10)

# Load gk_df
gk_path: Path =
gk_df: pd.DataFrame =

print("gk_df.info():")
gk_df.info()
print("##"*10)

## Remarks: Consolidate The Pandas DataFrame into One

In this section, we want to consolidate the scattered pandas dataframes loaded earlier into one unified dataframe for further analysis and processing. This consolidation would help us to understand the data distribution better whence we do Ecxploratory Data Analysis (EDA).

Notice that we need to convert the `upah` column from average hourly wage to average monthly wage to match the time period of the rest other columns.


In [ ]:
consol_df: pd.DataFrame = gk_df.merge(
    right =
    , how = "left"
    , on =
)

consol_df.rename(
    columns = {
        "periode" : "gk_periode_survei"
        , "jenis" : "jenis_pengeluaran"

    }
    , inplace = True
)

consol_df = consol_df.merge(
    right =
    , how = "left"
    , on =
)

consol_df = consol_df.merge(
    right =
    , how = "left"
    , on =
)

COUNT_WORKING_HOURS_PER_MONTH: int = 173 # unit: working_hourse / month

consol_df["upah"] = consol_df["upah"] * COUNT_WORKING_HOURS_PER_MONTH

print("consol_df.info():")
consol_df.info()
print("##"*10)
consol_df.head(n=10)

## Remarks: Data Pre-Processing - Convert String Columns into Categorical

In this section, we will convert string columns into appropriate categorical type

Furthermore, we also change the year (`tahun`) into categorical type. We do this to simplify the modeling as we would concentrate on building MLP model in this project.

We can revisit this case again next time once we learn about sequential deep learning, e.g., LSTM, RNN - which have better way to handle panel time data like this.

In [ ]:
# convert string columns into categorical

# convert all string columns into categorical data type
# convert the `tahun` column also into categorical data type
consol_df["..."] =

# ... put all your transformation here

consol_df.info()

## Remarks: Data Pre-Processing - Impute `Null` values with Median

Notice from the previous cell that the columns `gk`, `peng`, `ump`, and `upah` still have `null` values

Hence, in this section, we will impute these missing values with the median of the column.

Why median? Because economic metrics usually are skewed. Hence, by imputing using median, we would avoid its value from being affected by the outliers.

Furthermore, we define a `hierarchical_imputation()` function to perform this imputation hierarchically

The list of list in the `group_level` parameters should starts from more specific to more generic

For example

```Python
consol_df["gk"] = hierarchical_imputation(
    df = consol_df
    , column = "gk"
    , group_levels = [
        ["provinsi", "jenis_pengeluaran", "daerah", "tahun", "gk_periode_survei"]
        , ["provinsi", "jenis_pengeluaran", "daerah", "tahun"]
        , ["provinsi", "daerah", "tahun"]
        , ["provinsi", "tahun"]
    ]
)
```

you can do the same for other numeric columns


In [ ]:
def hierarchical_imputation(
        df: pd.DataFrame
        , column: str
        , group_levels: List[List[str]]
) -> pd.Series:
    """
    Impute missing values hierarchically using grouped medians.

    Fill missing values in a numeric column by applying median imputation
    across a sequence of grouping levels, from the most specific grouping
    to broader fallback groupings. Any remaining missing values after all
    grouping levels are filled with the overall median of the column.

    Parameters
    ----------
    df : pandas.DataFrame
        Input DataFrame containing the column to be imputed and the grouping
        columns used for hierarchical aggregation.
    column : str
        Name of the numeric column whose missing values will be imputed.
    group_levels : list of list of str
        Ordered list of grouping column combinations used for hierarchical
        imputation. Each inner list defines one grouping level. Earlier
        entries should be more specific, while later entries should be
        broader fallback groupings.

    Returns
    -------
    pandas.Series
        A Series containing the imputed values for the specified column,
        where missing values are filled using grouped medians first and the
        overall column median as the final fallback.
    """

    result: pd.DataFrame = df[column].copy()

    for each_group in group_levels:
        result = result.fillna(
            df.groupby(each_group, observed = True)[column].transform("median")
        )

    return result.fillna(df[column].median())

In [ ]:
# impute `null` values in numerical columns with median from respective groups

consol_df["..."] = hierarchical_imputation(

)

# ... put all your imputation here

print("consol_df.info():")
consol_df.info()
print("##"*10)
consol_df.head(n=10)

## Remarks: Data Pre-Processing - Exploratory Data Analysis (EDA)

Before moving further to decide what data to pre-process, we will take a look of the distribution of the data, and the correlation between the columns in our dataset. To do this, we will take advantage of the `ydata_profiling` library to generate an exploratory data analysis (EDA) report. This report will provide insights into the data distribution and correlations, helping us understand the dataset better.


In [ ]:
from ydata_profiling import ProfileReport

profile = ProfileReport(
    df = consol_df
    , title = "EDA Report"
)

profile.to_notebook_iframe()

From the EDA report above, we notice that (`gk`, `peng`) columns have high correlation factor (> 0.5). Similarly, we also notice that (`upah`, `ump`) columns have high correlation factor (> 0.5).

Hence, this means we can't use all four columns for our model, because involving all four columns will lead to multicollinearity, which can negatively impact the model's performance. Therefore, we need to decide which columns to use for our model.

Moving forward, we will use `upah / peng` ratio is a measurement of wellfare. This ratio allows us to retain the wellfare information, while avoiding the multicolinearity problem down the road

In [ ]:
# remove all the original columns: `gk`, `peng`, `ump`, `upah`
cols_to_drop: List[str] = ["gk", "peng", "ump", "upah"]
existing_cols: List[str] = [each_col for each_col in cols_to_drop if each_col in consol_df.columns]

if existing_cols:
    # compute `upah : peng` ratio as the new measure for wellfare to avoid multicolinearity problem down the road
    consol_df["upah_peng_ratio"] = consol_df["upah"]  / consol_df["peng"]

    # drop the original columns: `gk`, `peng`, `ump`, `upah`
    # ... put your code here

print("consol_df.info():")
consol_df.info()
print("##"*10)
consol_df.head(n=10)

## Remark: Data Pre-Processing - Convert Categorical Data into Multi One-Hot Encoded Format

In this section, we will conver the categorical columns into multi one-hot encoded format.

What this means practically is to convert all the unique values in the categories into its own column, and then assign a binary value to indicate whether the row contains that value or not.

In [ ]:
categ_cols: List[str] = consol_df.select_dtypes(include=["category"]).columns.to_list()

consol_df_encoded: pd.DataFrame = # ... put your conversion to multi-hot one-encoded here

print("consol_df_encoded.info():")
consol_df_encoded.info()
print("##"*10)

## Remark: Data Pre-Processing - Normalize the `upah_peng_ratio` Column

Next, we need to normalize the `upah_peng_ratio` column to ensure that it is on a similar scale as other numerical features, which can help in improving the stability of machine learning models.

In [ ]:
from sklearn.preprocessing import PowerTransformer

# ... put your transformation code here

consol_df_encoded["upah_peng_ratio_transformed"] = # ... and here

print("consol_df_encoded.info():")
consol_df_encoded.info()
print("##"*10)

In [ ]:
# Plot both `upah_peng_ratio` and `upah_peng_ratio_transformed`

%matplotlib inline

import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(
    nrows = 1
    , ncols = 2
    , figsize = (12, 6)
)

sns.histplot(
    data = consol_df_encoded["upah_peng_ratio"]
    , bins = 30
    , kde = True
    , ax = axes[0]
)
axes[0].set_title("upah_peng_ratio")

sns.histplot(
    data = consol_df_encoded["upah_peng_ratio_transformed"]
    , bins = 30
    , kde = True
    , ax = axes[1]
)
axes[1].set_title("upah_peng_ratio_transformed")

plt.tight_layout()
plt.show()

Notice how the original `upah_peng_ratio` column has a right-skewed distribution, while the transformed `upah_peng_ratio_transformed` column appears more symmetric and bell-shaped.

Furthermore, the range of values in the original `upah_peng_ratio` column is much larger than the range of values in the transformed `upah_peng_ratio_transformed` column. This shortened range is similar to the (0,1) range of the multi-hot encoded columns.

Both the normalization, and the tightened range in `upah_peng_ratio_transformed` are beneficial for deep learning models that (usually) assume normality and require values within a similar range.

In [ ]:
# Drop the original `upah_peng_ratio`
cols_to_drop: List[str] = ["upah_peng_ratio"]
existing_cols: List[str] = [each_col for each_col in cols_to_drop if each_col in consol_df_encoded.columns]

if existing_cols:
    # Drop the original `upah_peng_ratio` column
    # ... put your code here

print("consol_df_encoded.info():")
consol_df_encoded.info()
print("##"*10)

## Data Pre-Processing - Convert Pandas to Tensor and Split Dataset into Train : Validation : Test dataset

In this section, we will convert the processed Pandas DataFrame into Numpy Tensor format, and split the dataset into training, validation, and testing subsets for model training and evaluation.

Mathematically, we can imagine a Tensor as high-dimension metrics, where each dimensions represents a feature in the dataset. In our case, each dimension in Tensor represent the entries in multi-hot encoded columns, and one target numerical `upah_peng_ratio_transformed`.

Furthermore, we will conduct a brief 'experiment' whilst building a model.

Why do we need to experiment?

The need for experimenting, or trying several different configurations in deep learning is important, because many of the decision choices are usually dictated by the context and the problem at hand. That is, there is "Physics" rule that guide the choice.

Hence, in absence of these fundamental laws like those in hard-science, data scientists usually need to choose the best configuration from several candidates - and this is where experimentation comes in handy.

We will experiment with different model architectures, hyperparameters, and optimization techniques to find the best configuration for our model. This process is iterative and requires careful consideration of the trade-offs between model performance and computational resources.

In [ ]:
import numpy as np

target_col: List[str] = ["upah_peng_ratio_transformed"]
existing_cols: List[str] = [each_col for each_col in consol_df_encoded.columns if each_col in target_col]

if existing_cols:
    # Splitting the consol_df_encoded into features and target
    # In our case, the features are the result from multi-hot one-encoded column
    # whilst the target would be the `upah_peng_ratio_transformed` column
    upr_ds: pd.Series = consol_df_encoded.pop("upah_peng_ratio_transformed")

print("upr_ds.info():")
consol_df_encoded.info()
print("##"*10)
print(f"upr_ds.shape: {upr_ds.shape}")
upr_ds.head(n=10)
print("##"*10)

# Here, we can directly convert the Pandas to Numpy, because we already deal with the `null` values and multi-hot encoding
# in the above section using Pandas
# We are creating Mathematical Tensor in Python using `np.ndarray`
feat_np: np.ndarray = consol_df_encoded.to_numpy()
tgt_np: np.ndarray = upr_ds.to_numpy()

# Inspecting the resulting Numpy array
print("feat_np info:")
print(f"shape: {feat_np.shape}")
print(f"dtype: {feat_np.dtype}")
print(f"ndim: {feat_np.ndim}")
print(f"size: {feat_np.size}")
print("##"*10)

print("tgt_np info:")
print(f"shape: {tgt_np.shape}")
print(f"dtype: {tgt_np.dtype}")
print(f"ndim: {tgt_np.ndim}")
print(f"size: {tgt_np.size}")
print("##"*10)

In this section, we will split the original Tensor dataset into smaller `train / val / test` Tensor dataset.

As noted, for this, we would make use of the `train_test_split` function from `sklearn.model_selection` module. This function allows us to split the dataset into training, validation, and testing sets with specified proportions and random state for reproducibility.

For our case, we would use a `train : val : test = 80 : 10 : 10` split. We achieve this by splitting the dataset in two-rounds: in the first round, we split the original dataset into `train : temp = 80 : 20` split; in the second round, we then split the temp dataset into `val : test = 50 : 50` split.

In [ ]:
from sklearn.model_selection import train_test_split

# Creating First Split: train / temp
X_train, X_temp, y_train, y_temp = train_test_split(
    feat_np
    , tgt_np
    , test_size = 0.2
    , random_state = 42
)

# Creating Second Split: val / test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp
    , y_temp
    , test_size = 0.5
    , random_state = 42
)

## Remark: Building Multi-Layer Perceptron (MLP) Model
### Variation 01: Two-Hidden Layer MLP

Now, after splitting our dataset into `train : validation : test` groups, we proceed to build our MLP model.

We will create two variations: (1) one architecture is composed of two hidden-layers; and (2) another architecture is composed of four hidden layers. In that way, we can see how adding more layers (complexity) affects the model's final behavior.

In [ ]:
# Building `model01` - a simpler architecture with only two-hidden layers
import tensorflow as tf

model01: tf.keras.Sequential = # ... put your code for model definition here

model01.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate = 1e-4)
    , loss = tf.keras.losses.MeanSquaredError()
    , metrics = [tf.keras.metrics.MeanSquaredError(name="rmse")]
)

In [ ]:
SEED: int = 42
EPOCHS: int = 100
BATCH_SIZE: int = 32

# Training `model01`
model01_history: tf.keras.callbacks.History = # ... put your training code here

In [ ]:
from matplotlib import pyplot as plt

# Create small utility function to plot Loss Evolution
def plot_loss(
        history: tf.keras.callbacks.History
        , title: str = "Loss Evolution"
        , x_label: str = "Epoch"
        , y_label: str = "Loss"
) -> None:
    fig, ax = plt.subplots(figsize = (14, 5))

    plt.plot(
        history.history["loss"]
        , label = "Training Loss"
    )

    if "val_loss" in history.history:
        plt.plot(
            history.history["val_loss"]
            , label = "Validation Loss"
            , linewidth = 2
        )

    ax.set_title(label = title)
    ax.set_xlabel(xlabel = x_label)
    ax.set_ylabel(ylabel = y_label)
    ax.legend()

    plt.tight_layout()
    plt.show()


In [ ]:
%matplotlib inline

plot_loss(
    history = model01_history
    , title = f"Loss Evolution for {model01.name}"
)

### Variation 02: Four-Hidden Layer MLP

In this alternative model `model02`, we will try with four hidden MLP layers.

Compared to `model01`, the `model02` has more nodes and more complexity. From what we have learnt so far, this so-called "more advanced" is expected to give better prediction compared to the simpler `model01` thanks to the more numerous count of nodes.

By comparing the two models, we will see if this hypothesis holds

In [ ]:
model02: tf.keras.Sequential = # ... put your code for model definition here

model02.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate = 1e-4)
    , loss = tf.keras.losses.MeanSquaredError()
    , metrics = [tf.keras.metrics.MeanSquaredError(name="rmse")]
)

In [ ]:
SEED: int = 42
EPOCHS: int = 100
BATCH_SIZE: int = 32

# Training `model01`
model02_history: tf.keras.callbacks.History = # .. put your model training code here

In [ ]:
%matplotlib inline

plot_loss(
    history = model02_history
    , title = f"Loss Evolution for {model02.name}"
)

## 🧠Food for Thought

Question:

After training both `model01` (which has two-hidden layers) and `model02` (which has four-hidden layers), which model do you think will perform better on unseen data? Why?

## Remark: Experimenting with Hyper Parameters

From comparing different layers, we can have better understanding about how model architecture affect the model's behavior in both training, as well as in production.

Now, we will switch our attention to one hyperparameter that we have encountered in session02, namely: the `learning_rate`.

We will use `model01` for this experiment, and varies solely the `learning_rate` hyper-parameter

In [ ]:
import random

from matplotlib import pyplot as plt
import numpy as np
import tensorflow as tf

SEED: int = 42
EPOCHS: int = 100
BATCH_SIZE: int = 32

learning_rates: List[float] = sorted([
    1e-3
    , 1e-4
    , 1e-5
    , 1e-6
])

def reset_random_seed(
        my_seed: int = 42
) -> None:
    random.seed(my_seed)
    np.random.seed(my_seed)
    tf.random.set_seed(my_seed)


def build_model01(
        learning_rate: float = 1e-4
) -> tf.keras.Sequential:
    model01: tf.keras.Sequential = # ... put your code for defining model here

    model01.compile(
        optimizer = tf.keras.optimizers.Adam(learning_rate = learning_rate)
        , loss = tf.keras.losses.MeanSquaredError()
        , metrics = [tf.keras.metrics.MeanSquaredError(name="rmse")]
    )

    return model01

In [ ]:
from typing import Dict

histories: Dict = {}
trained_models: Dict = {}

for each_rate in learning_rates:

    print(f"Training model01 with learning_rate: {each_rate}")

    reset_random_seed(SEED)
    model01: tf.keras.Sequential = build_model01(learning_rate = each_rate)

    history: tf.keras.callbacks.History = # ... put your code to train model here

    histories[str(each_rate)] = history
    trained_models[str(each_rate)] = model01

In [ ]:
%matplotlib inline

fig, axs = plt.subplots(
    nrows = 1
    , ncols = 2
    , figsize = (14, 5)
)

for each_lr, each_history in histories.items():

    axs[0].plot(
        each_history.history["loss"]
        , label = f"Learning Rate: {each_lr}"
    )

    axs[1].plot(
        each_history.history["val_loss"]
        , label = f"Learning Rate: {each_lr}"
    )


axs[0].set_title("Training Loss Evolution")
axs[0].set_xlabel("Epoch")
axs[0].set_ylabel("MSE Loss")
axs[0].legend()

axs[1].set_title("Validation Loss Evolution")
axs[1].set_xlabel("Epoch")
axs[1].set_ylabel("MSE Loss")
axs[1].legend()

plt.tight_layout()
plt.show()

## 🧠 Food for Thought

Question:

So, what do you think about the impact of learning rate on model training performance? How does this connects to what he have learnt thus far?